## Lakebase Verification Queries

This notebook runs the verification queries from `sql/04_verify_embeddings.sql` directly against the Lakebase Postgres instance.

**Why?** `.sql` files in the Databricks editor execute on the SQL warehouse, which cannot connect to Lakebase Postgres. This notebook uses the Databricks SDK to generate an OAuth token and `psycopg` to run the queries natively against the Postgres endpoint.

In [0]:
# No additional packages needed:
# - psycopg2 is pre-installed on this compute
# - pandas is pre-installed
# - dbutils.secrets is built-in

In [0]:
import psycopg2
import pandas as pd

# Connection via massive_app role (username + password from secrets)
LAKEBASE_URL = dbutils.secrets.get(scope="database", key="lakebase-url")

conn = psycopg2.connect(LAKEBASE_URL)
print(f"Connected to Lakebase as massive_app")

Connected to Lakebase as massive_app


In [0]:
query1 = """
SELECT 'ticker_news_documents' AS table_name, COUNT(*) AS total, NULL::bigint AS embedded
FROM ticker_news_documents
UNION ALL
SELECT 'ticker_news_embeddings', COUNT(*), COUNT(embedding)
FROM ticker_news_embeddings
UNION ALL
SELECT 'ticker_news_chunk_embeddings', COUNT(*), COUNT(embedding)
FROM ticker_news_chunk_embeddings;
"""

df1 = pd.read_sql(query1, conn)
print("=== Row Counts ===")
display(df1)

/home/spark-42f7cf07-9947-406b-8f40-8f/.ipykernel/72/command-5378904804046300-2593962941:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)


=== Row Counts ===


table_name,total,embedded
ticker_news_documents,17,null
ticker_news_embeddings,0,0.0
ticker_news_chunk_embeddings,0,0.0


In [0]:
query2 = """
SELECT
    'ticker_news_embeddings' AS table_name,
    pg_typeof(embedding)::text AS column_type,
    vector_dims(embedding) AS dims,
    model_name
FROM ticker_news_embeddings
LIMIT 1;
"""

df2 = pd.read_sql(query2, conn)
print("=== Vector Type Check ===")
if df2.empty:
    print("No embeddings found yet — run the embedding notebook first.")
else:
    display(df2)

=== Vector Type Check ===
No embeddings found yet — run the embedding notebook first.


/home/spark-42f7cf07-9947-406b-8f40-8f/.ipykernel/72/command-5378904804046301-3848431003:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)


In [0]:
query3 = """
SELECT tablename, indexname, indexdef
FROM pg_indexes
WHERE schemaname = 'public'
  AND indexname LIKE '%embedding%'
ORDER BY tablename;
"""

df3 = pd.read_sql(query3, conn)
print("=== HNSW Index Check ===")
display(df3)

/home/spark-42f7cf07-9947-406b-8f40-8f/.ipykernel/72/command-5378904804046302-2023820566:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)


=== HNSW Index Check ===


tablename,indexname,indexdef
ticker_news_chunk_embeddings,ticker_news_chunk_embeddings_pkey,CREATE UNIQUE INDEX ticker_news_chunk_embeddings_pkey ON public.ticker_news_chunk_embeddings USING btree (id)
ticker_news_chunk_embeddings,idx_ticker_news_chunk_embeddings_embedding,CREATE INDEX idx_ticker_news_chunk_embeddings_embedding ON public.ticker_news_chunk_embeddings USING hnsw (embedding vector_cosine_ops)
ticker_news_embeddings,ticker_news_embeddings_pkey,CREATE UNIQUE INDEX ticker_news_embeddings_pkey ON public.ticker_news_embeddings USING btree (id)
ticker_news_embeddings,idx_ticker_news_embeddings_embedding,CREATE INDEX idx_ticker_news_embeddings_embedding ON public.ticker_news_embeddings USING hnsw (embedding vector_cosine_ops)


In [0]:
query4 = """
SELECT
    d.ticker,
    d.title,
    (e.embedding <=> (SELECT embedding FROM ticker_news_embeddings LIMIT 1))::numeric(10,4) AS cosine_distance
FROM ticker_news_embeddings e
JOIN ticker_news_documents d ON d.id = e.id
ORDER BY cosine_distance
LIMIT 5;
"""

# Reconnect if the connection was dropped (idle timeout)
try:
    conn.cursor().execute("SELECT 1")
except Exception:
    conn = psycopg2.connect(dbutils.secrets.get(scope="database", key="lakebase-url"))

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    df4 = pd.read_sql(query4, conn)
print("=== Cosine Similarity Smoke Test (Top 5 nearest) ===")
if df4.empty:
    print("No embeddings found yet — run the embedding notebook first.")
else:
    display(df4)

=== Cosine Similarity Smoke Test (Top 5 nearest) ===
No embeddings found yet — run the embedding notebook first.


In [0]:
conn.close()
print("Connection closed.")